In [ ]:
##
#Import Packages
import numpy as np
import sys
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
##
#Define Path to Code Database
DirPath = '/PATH/TO/bin/'

#Define Path to Example Data
DataPath = '/PATH/TO/ExampleData/'

#Define Output Path
OutputPath = '/PATH/TO/Output/'

In [ ]:
##
#Import Custom Functions
sys.path.append(DirPath)
from ImportData import *
from FreedAnalytical import *

In [ ]:
##
#Define Default Properties

##
#Define Diffusion Tensor (mm/s)
D = 0.0001

##
#Define Relaxation Times & B1 (Mean Values in Post-Mortem Brain)
T1 = 650
T2 = 35
B1 = 1

##
#Define Range Relaxation Times, B1, and D
T1Range = np.arange(100,1200+1100/100,1100/100)
T2Range = np.arange(10,80+70/100,70/100)
B1Range = np.arange(0.1,1.2+1.1/100,1.1/100)
DRange = np.arange(0,0.000201,0.000002)



In [ ]:
##
#Load Data
    # bvecs - bvectors (3xn)
    # FlipAngles - Flip Angles (degrees) (1xn)
    # tau - Diffusion Gradient Duration (seconds) (1xn), 
    # G - Diffusion Gradient Duration (G/cm - Equivalent to mT/m Divided by 10) (1xn)
    # TRs - Repetition Times (seconds) (1xn)
    # b0s - Array Defining b0 locations (b0 = 1, dwi = 0) (1xn)

bvecs, FlipAngles, tau, G, TRs, b0s = ImportTextDataDWSSFP(DataPath)

In [ ]:
##
#Simulate Signals

##
#Initialise Arrays
S_T1 = np.zeros(T1Range.shape[0])
S_T2 = np.zeros(T2Range.shape[0])
S_B1 = np.zeros(B1Range.shape[0])
S_D = np.zeros(B1Range.shape[0])
S0_T1 = np.zeros(T1Range.shape[0])
S0_T2 = np.zeros(T2Range.shape[0])
S0_B1 = np.zeros(B1Range.shape[0])
S0_D = np.zeros(B1Range.shape[0])


##
#Forward Simulate Comparison Signal
for k in range(B1Range.shape[0]):
    S_T1[k] = FreedDWSSFP(G[0], tau[6], TRs[0], FlipAngles[0]*B1, D, T1Range[k], T2)
    S_T2[k] = FreedDWSSFP(G[0], tau[6], TRs[0], FlipAngles[0]*B1, D, T1, T2Range[k])
    S_B1[k] = FreedDWSSFP(G[0], tau[6], TRs[0], FlipAngles[0]*B1Range[k], D, T1, T2)
    S_D[k] = FreedDWSSFP(G[0], tau[6], TRs[0], FlipAngles[0]*B1, DRange[k], T1, T2)
    S0_T1[k] = FreedDWSSFP(G[0], tau[6], TRs[0], FlipAngles[0]*B1, 0, T1Range[k], T2)
    S0_T2[k] = FreedDWSSFP(G[0], tau[6], TRs[0], FlipAngles[0]*B1, 0, T1, T2Range[k])
    S0_B1[k] = FreedDWSSFP(G[0], tau[6], TRs[0], FlipAngles[0]*B1Range[k], 0, T1, T2)
    S0_D[k] = FreedDWSSFP(G[0], tau[6], TRs[0], FlipAngles[0]*B1, 0, T1, T2)

In [ ]:
##
#Plot Signals
fig = plt.figure()
idx = np.flatnonzero(b0s)[-1]
plt.plot(S_T1/S0_T1,linewidth=2)
plt.plot(S_T2/S0_T2,linewidth=2)
plt.plot(S_B1/S0_B1,linewidth=2)
plt.plot(S_D/S0_D,linewidth=2)
plt.yticks([0,1])
plt.ylim([0,1])
plt.xticks([0,S_T1.shape[0]],labels=['Low','High'],size=12)
plt.title('DW-SSFP Signal Dependencies', size=12)
plt.ylabel('Diffusion Attenuation (Normalised)', size=12)
plt.xlabel('$T_{1}$/$T_{2}$/$B_{1}$/D', size=12)
plt.legend(['$T_{1}$','$T_{2}$','$B_{1}$','$D$']),

In [ ]:
#Save Figure
fig.savefig(''.join([OutputPath,'Figure2b.pdf']),dpi=300,format='pdf',bbox_inches='tight')